In [1]:
# 1. Imports and Setup
import pandas as pd
import shap
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

# Make plots render inline
%matplotlib inline

C:\Users\PC\Desktop\Fraud-Detector\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Increase plot size for readability
plt.rcParams["figure.figsize"] = (10, 6)

In [3]:
# 2. Load Models
# =============================
print("Loading trained models...")
model_fraud = joblib.load("../models/xgb_fraud_model.pkl")
model_card = joblib.load("../models/xgb_credit_model.pkl")

Loading trained models...


In [4]:
# 3. Load Datasets
# =============================
print("Loading processed datasets...")
X_fraud = pd.read_csv("../data/processed/fraud_data_processed.csv")
X_card = pd.read_csv("../data/creditcard.csv")

Loading processed datasets...


In [5]:
# 4. Encode Object Columns
# =============================
print("Encoding categorical columns...")

for col in X_fraud.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_fraud[col] = le.fit_transform(X_fraud[col].astype(str))

for col in X_card.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_card[col] = le.fit_transform(X_card[col].astype(str))

Encoding categorical columns...


In [6]:
# 5. Use Smaller Samples for SHAP
# =============================
# SHAP can be very heavy on full datasets → sample for speed
fraud_sample = X_fraud.sample(n=2000, random_state=42) if len(X_fraud) > 2000 else X_fraud
card_sample = X_card.sample(n=2000, random_state=42) if len(X_card) > 2000 else X_card


In [8]:
# 6. Fraud Model Explainability
# =============================
print("Explaining Fraud Detection Model...")

# Ensure all features are numeric (drop or encode object columns)
fraud_sample_numeric = fraud_sample.copy()

# Drop non-numeric columns
fraud_sample_numeric = fraud_sample_numeric.select_dtypes(include=["int64", "float64"])

# Alternatively: encode categorical if you want to keep them
# fraud_sample_numeric = pd.get_dummies(fraud_sample, drop_first=True)

# Run SHAP with cleaned features
explainer_fraud = shap.Explainer(model_fraud, fraud_sample_numeric)
shap_values_fraud = explainer_fraud(fraud_sample_numeric)

# Beeswarm Plot
shap.summary_plot(shap_values_fraud, fraud_sample_numeric)

# Bar Plot
shap.plots.bar(shap_values_fraud)


Explaining Fraud Detection Model...


ExplainerError: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. This check failed because for one of the samples the sum of the SHAP values was -9.031543, while the model output was -4.544255. If this difference is acceptable you can set check_additivity=False to disable this check.

In [ ]:
# 7. Credit Card Model Explainability
# =============================
print("Explaining Credit Card Fraud Model...")

explainer_card = shap.Explainer(model_card, card_sample)
shap_values_card = explainer_card(card_sample)

# Beeswarm Plot
shap.summary_plot(shap_values_card, card_sample)

# Bar Plot
shap.plots.bar(shap_values_card)

print("✅ SHAP explainability complete. Plots are displayed above.")